# 4. Elastic net and group lasso: using gene structure

**Learning goals**: the elastic net's grouping effect and whether it matters on real data, and the
group lasso on gene modules, with a from-scratch solver.

Genes work in programmes: dozens of genes rise and fall together. The lasso picks one member of
each programme; the elastic net ($\alpha\lVert\beta\rVert_1 + \frac{1-\alpha}{2}\lVert\beta\rVert_2^2$)
tends to keep the whole correlated set. The group lasso goes further: it is told the groups and
keeps or drops each one as a unit.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
rng = np.random.default_rng(7)

In [2]:
# The project's data: one drug, its screened cell lines, 2,000 most variable genes
from drugresponse.experiment import Store
from drugresponse.models import Design
S = Store()
drug_name = "Nutlin-3a (-)"
d = S.drugs.index[S.drugs.drug_name == drug_name][0]
ok = S.R[d].notna().values
des = Design(2000).fit(S.Ev[ok], S.tissue[ok], S.sanger[ok])
X = des.transform(S.Ev[ok], S.tissue[ok], S.sanger[ok])[:, ~des.is_tissue()]   # genes only
y = S.R[d].values[ok]
genes = S.genes[des.genes_]
print(f"{drug_name}: {X.shape[0]} cell lines x {X.shape[1]} genes")

Nutlin-3a (-): 943 cell lines x 2000 genes


In [3]:
from sklearn.linear_model import ElasticNet, Lasso
Xc, yc = X - X.mean(0), y - y.mean()
# How does the weight spread as the penalty moves from lasso (1.0) towards ridge?
# Each mixing value tuned by 3-fold CV; "effective genes" = (sum |b|)^2 / sum b^2,
# the number of equally weighted genes that would give the same spread.
from sklearn.linear_model import ElasticNetCV
rows = []
for l1 in [1.0, 0.5, 0.2, 0.05]:
    m = ElasticNetCV(l1_ratio=l1, cv=3, alphas=20, max_iter=5000, tol=1e-3).fit(Xc, yc)
    b = m.coef_
    eff = (np.abs(b).sum() ** 2) / max((b ** 2).sum(), 1e-12)
    top5 = np.sort(np.abs(b))[::-1][:5].sum() / max(np.abs(b).sum(), 1e-12)
    rows.append((l1, int((b != 0).sum()), round(eff, 1), round(top5, 2), round(-m.mse_path_.mean(1).min(), 5)))
pd.DataFrame(rows, columns=["l1_ratio", "genes selected", "effective genes", "weight share of top 5", "-CV MSE"])

,l1_ratio,genes selected,effective genes,weight share of top 5,-CV MSE
0,1.00,126,43.8,0.25,-0.00351
1,0.50,126,44.0,0.25,-0.00351
2,0.20,126,44.4,0.25,-0.00351
3,0.05,134,46.2,0.25,-0.00350


On this drug the mixing weight hardly matters. Once each setting is tuned by CV, the lasso and
the elastic net select about the same number of genes and spread weight the same way. The L2 part
only changes the solution when it is large relative to the L1 part, and CV prefers penalties too
small for that. This matches the main results: across 281 drugs the lasso and the elastic net have
the same median r (0.382). The textbook grouping effect is real, but it is not guaranteed to matter
on a given dataset, so check it rather than assume it.

## Group lasso on co-expression modules
The project clusters genes into 100 modules by their expression profiles (k-means on standardised
gene vectors, which is clustering on correlation) inside each training fold, then penalises
$\lambda \sum_g \sqrt{p_g}\,\lVert\beta_g\rVert_2$. The solver is accelerated proximal gradient (FISTA),
written from scratch in `drugresponse.models.group_lasso_path`.

In [4]:
from drugresponse.models import coexpression_modules, group_lasso_path, group_lambda_max
groups = coexpression_modules(Xc, 100)
lmax = group_lambda_max(Xc, yc, groups)
path = group_lasso_path(Xc, yc, groups, lmax * np.logspace(0, -1.5, 8), max_iter=3000, tol=1e-6)
for lam_frac, b in zip(np.logspace(0, -1.5, 8), path):
    active = np.unique(groups[np.abs(b) > 0])
    print(f"lambda = {lam_frac:.3f} x max: {len(active):3d} modules active, {np.sum(np.abs(b) > 0):4d} genes")

lambda = 1.000 x max:   0 modules active,    0 genes
lambda = 0.611 x max:   3 modules active,  129 genes
lambda = 0.373 x max:   5 modules active,  157 genes
lambda = 0.228 x max:   9 modules active,  275 genes
lambda = 0.139 x max:  20 modules active,  356 genes
lambda = 0.085 x max:  32 modules active,  615 genes
lambda = 0.052 x max:  51 modules active, 1046 genes
lambda = 0.032 x max:  69 modules active, 1343 genes


In [5]:
b = path[-3]
act = pd.Series(np.abs(b), index=genes).groupby(groups).sum().sort_values(ascending=False)
for g in act.index[:3]:
    members = genes[groups == g]
    top = pd.Series(b[groups == g], index=members).abs().sort_values(ascending=False).index[:8]
    print(f"module {g} ({len(members)} genes): {', '.join(top)}")

module 18 (16 genes): CDKN1A, LINC00973, ALCAM, CAVIN3, G0S2, MT1X, CA9, MT1E
module 59 (27 genes): PURPL, ETS1, CD70, RFTN1, RUNX3, SLAMF7, MEF2C, SRGN
module 1 (28 genes): CDKN2A, CKB, ASRGL1, CENPV, NEFH, NETO2, TSPYL5, CLGN


A module is selected or not as a whole. The output reads like a short list of biological programmes
rather than a scattered list of genes. The cost is speed: the group lasso is solved with an iterative
proximal method and tuned on a 12-point path, so the project runs it on the 36-drug focus panel only.

**Key takeaways**
* Elastic net: sparse, and in principle shares weight across correlated genes; on this data, at
  CV-chosen penalties, it behaves almost exactly like the lasso.
* Group lasso: selection at the level of a module or pathway; within a module, ridge-like shrinkage.
* Groups can come from biology (curated pathways) or from the data (co-expression), as here.